Pytorch - QuickStart tutorial
1. Two primitives to work with data - DataLoader and Dataset
2. Dataset - Stores the samples and their corresponding labels
3. Dataloaders - wraps and iterable around the Dataset.

torch.utils.data.DataLoader/Dataset



In [1]:
import torch

In [2]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2 # Convert raw PIL into tensor format the model expects (PIL - Python Image Library)


1. Pytorch offers domain specific which include datasets :  TorchText, TorchVision, and TorchAudio 
    1.1 using torchvision in our example now.

In [3]:
# Downloading Test and Training datasets

training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
)

In [4]:
#Pass the Dataset as an arguement to the Dataloader, to wrap an iterable over our dataset. 
#Supports Automatic Batching, sampling, shuffling and multiprocessing data loading. We also define a batch size now of 64.
#To split the dataset into batches of 64.

batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for x, y in test_dataloader:
    print(f"Shape of X [N, C, H,W]: {x.shape}") # N - C - H - W : Batch sample size - Channel (greyscale - 1, RGB - 3) - Height - Width
    print(f"Shape of Y [N,C,H,W] : {y.shape}")
    break

Shape of X [N, C, H,W]: torch.Size([64, 1, 28, 28])
Shape of Y [N,C,H,W] : torch.Size([64])


Creating Models
1. To define a neural network in pytorch :
    1.1. Create a class that inhertis from the nn.Module.
    1.2. Define the layers in the __init__ function.
    1.3. Specify how the data passes through the network in the forward() function
2. In order to accelerate the operations in the network, move it to the accelator (CUDA) if present, else use CPU.

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using {device} device")


using cuda device


In [8]:

# Define Model - class

class NeuralNetwork(nn.Module): # class classname(<inheriting from>)
    def __init__(self):
        super().__init__() # super class (from where inherited from)
        self.flatten = nn.Flatten() # - collapses a Multidimentional array (contiguous range) into one dimention.(one vector)
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512), # (in_features, out_features)
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )
    
    def forward(self,x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    



Creating the object of the nn to use.

In [9]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Optimizing the Model Parameters
1. To Train a model, we need a loss function and an optimizer.


In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In a single training loop, the model makes predictions on the training dataset, and backpropagates the prediction error to adjust the model parameters.

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train() # Enterning the training mode.
    for batch, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute Prediction Error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss : {loss:>7f} [{current:>5d}/{size:>5d}]")
    

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() # Inference mode
    test_loss, correct = 0, 0
    with torch.no_grad(): # wtf is this.? 
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # what shit is this.?
    test_loss /= num_batches
    correct /= size
    print(f"Test Error : \n Accuracy: {(100*correct):>0.1f}%, Avg_loss : {test_loss:>8f} \n")


The training process is carried over multiple epochs(iterations) --> model weights are learned during training and to make better predictions.

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epochs {t+1} \n -----------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done.!")

Saving Models
1. A common way to save a model is to serialize the internal State dictionary

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Loading Models
1. Re-creating the model structure and loading the state dictionary into it.

In [11]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only = True))

<All keys matched successfully>

In [12]:
# Testing to make Predictions

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
